# 延迟-多普勒域表示

**Delay-Doppler Representations**

本notebook介绍延迟-多普勒（Delay-Doppler, DD）信号表示的基本概念，这是理解OTFS（正交时频空）调制技术的核心基础。

## 1. 目标

通过本notebook的学习，你将：

- 理解延迟（Delay）和多普勒（Doppler）的物理意义
- 理解延迟-多普勒域与时域/频域的关系
- 为理解OTFS的核心创新打下基础

## 2. 延迟-多普勒域直观理解

### 2.1 雷达类比

延迟-多普勒域最直观的理解方式是借助**雷达系统**的类比：

| 参数 | 物理意义 | 雷达类比 |
|------|----------|----------|
| **延迟 τ (Delay)** | 信号往返时间，对应目标距离 | 雷达发射脉冲，接收回波的时间差 |
| **多普勒 ν (Doppler)** | 目标速度引起的频率偏移 | 运动目标使回波产生频率偏移 |

### 2.2 为什么这种表示匹配无线信道物理？

无线信道中的信号传播遵循以下物理规律：

1. **多径传播**：信号从发射端到接收端经历多条路径，每条路径由不同的反射器产生
2. **延迟**：不同路径长度不同，导致信号到达时间不同（延迟的物理来源）
3. **多普勒效应**：移动的反射器（如高速列车、无人机）会导致信号频率偏移
4. **稀疏性**：实际无线信道通常只有少数几个主导反射器（稀疏特性）

延迟-多普勒域表示直接刻画了这些物理现象，因此是描述无线信道最自然的方式。

## 3. 延迟-多普勒信道的物理意义

### 3.1 无线信道 = 镜面反射器的集合

无线信道可以建模为一系列**镜面反射器**（specular reflectors）的集合：

- 每个反射器可以是**静止的**或**运动的**
- 静止反射器引入纯延迟（无多普勒）
- 运动反射器引入延迟 + 多普勒频移

### 3.2 信道冲激响应 h(τ, ν)

延迟-多普勒域的信道冲激响应记为 **h(τ, ν)**，其物理含义为：

- **τ（延迟）**：信号沿某条路径传播的往返时间
- **ν（多普勒）**：该路径上的多普勒频率偏移
- **h(τ, ν)**：具有特定延迟τ和多普勒ν的反射器集群的复增益

每个延迟-多普勒 tap 代表一组具有相同延迟和多普勒特性的反射器集群。

## 4. 代码演示：创建延迟-多普勒信道响应

下面我们用代码创建一个典型的延迟-多普勒信道，并将其可视化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set up Chinese font support
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------------------------------------------------
# Create a delay-Doppler channel with 3 paths
# Path 1: Direct LOS - delay=0us, Doppler=0Hz
# Path 2: Static reflector - delay=1us, Doppler=0Hz
# Path 3: Moving reflector - delay=2us, Doppler=500Hz
# ---------------------------------------------------

# Define grid parameters
delay_max = 5  # Maximum delay in us
doppler_max = 1000  # Maximum Doppler in Hz
n_delay = 500  # Number of delay samples
n_doppler = 400  # Number of Doppler samples

# Create delay and Doppler axes
delays = np.linspace(0, delay_max, n_delay)  # delay axis in us
dopplers = np.linspace(-doppler_max, doppler_max, n_doppler)  # Doppler axis in Hz

# Initialize sparse 2D channel matrix h(tau, nu)
h = np.zeros((n_delay, n_doppler), dtype=np.complex128)

# Path parameters: (delay_idx, doppler_idx, amplitude)
# Path 1: Direct LOS at tau=0, nu=0 (center of grid)
path1_delay_idx = 0
path1_doppler_idx = n_doppler // 2  # index for 0 Hz
path1_amp = 1.0

# Path 2: Static reflector at tau=1us, nu=0
path2_delay_idx = int(1 / delay_max * n_delay)  # 1us position
path2_doppler_idx = n_doppler // 2  # index for 0 Hz
path2_amp = 0.5

# Path 3: Moving reflector at tau=2us, nu=500Hz
path3_delay_idx = int(2 / delay_max * n_delay)  # 2us position
path3_doppler_idx = int((500 + doppler_max) / (2 * doppler_max) * n_doppler)  # 500Hz position
path3_amp = 0.7

# Place the three paths in the channel matrix
h[path1_delay_idx, path1_doppler_idx] = path1_amp
h[path2_delay_idx, path2_doppler_idx] = path2_amp
h[path3_delay_idx, path3_doppler_idx] = path3_amp

# Visualize the delay-Doppler channel as a 2D heatmap
fig, ax = plt.subplots(figsize=(10, 6))

# Display channel magnitude
im = ax.imshow(np.abs(h), aspect='auto', origin='lower',
               extent=[-doppler_max, doppler_max, 0, delay_max],
               cmap='hot', interpolation='nearest')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('|h(tau, nu)|', fontsize=12)

# Label axes: x-axis = Doppler (Hz), y-axis = Delay (us)
ax.set_xlabel('Doppler (Hz)', fontsize=12)
ax.set_ylabel('Delay (us)', fontsize=12)
ax.set_title('Delay-Doppler Channel Response h(tau, nu)', fontsize=14)

# Add annotations for the three paths
ax.annotate('Path 1: LOS\n(tau=0, nu=0)', xy=(0, 0), xytext=(-400, 0.8),
            fontsize=10, color='white', ha='center',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

ax.annotate('Path 2: Static\n(tau=1us, nu=0)', xy=(0, 1), xytext=(-400, 1.8),
            fontsize=10, color='white', ha='center',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

ax.annotate('Path 3: Moving\n(tau=2us, nu=500Hz)', xy=(500, 2), xytext=(700, 3),
            fontsize=10, color='white', ha='center',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

plt.tight_layout()
plt.show()

print("Delay-Doppler channel created successfully!")
print(f"Path 1 (LOS): delay=0us, Doppler=0Hz, amplitude={path1_amp}")
print(f"Path 2 (Static): delay=1us, Doppler=0Hz, amplitude={path2_amp}")
print(f"Path 3 (Moving): delay=2us, Doppler=500Hz, amplitude={path3_amp}")

## 5. 延迟-多普勒域的性质

### 5.1 镜面反射器的稀疏表示

在延迟-多普勒域中：

- **镜面反射器表现为离散的点/峰值**
  - 每个反射器在 h(τ, ν) 平面占据一个唯一的位置 (τ, ν)
  - 该位置的幅度对应反射器的复增益（幅度和相位）

- **实际信道的稀疏性**
  - 实际无线信道通常只有少数几个主导反射器
  - 大部分延迟-多普勒网格位置的能量为零或接近零
  - 这种稀疏性使得信道表示非常高效

### 5.2 与时域/频域表示的对比

| 表示域 | 变量 | 物理含义 | 特点 |
|--------|------|----------|------|
| 时域 | t | 时间 | 直观但难以捕捉频率弥散 |
| 频域 | f | 频率 | 显示频率成分但难以捕捉时变特性 |
| 延迟-多普勒域 | (τ, ν) | 延迟与多普勒 | 直接刻画信道物理特性 |

### 5.3 信道稀疏性的优势

由于实际无线信道在延迟-多普勒域是稀疏的：

1. **信道估计变得简单** - 只需估计少数几个显著路径
2. **压缩感知技术可用** - 利用稀疏性减少导频开销
3. **信道表示更高效** - 只需要存储几个非零tap的位置和增益

## 6. 关联OTFS

### 6.1 OTFS：把QAM符号放在延迟-多普勒网格上

OTFS（正交时频空）的核心创新在于：

- **传统OFDM**：将QAM符号放置在**时频网格**上（时间-频率二维平面）
- **OTFS**：将QAM符号放置在**延迟-多普勒网格**上（延迟-多普勒二维平面）

这一改变带来了巨大的优势：

### 6.2 信道卷积变为简单乘法

在延迟-多普勒域中，信道对信号的作用变得极其简单：

- **时域/频域**：信道作用是复杂的卷积运算，涉及时间和频率的弥散
- **延迟-多普勒域**：信道作用简化为**对每个QAM符号的复增益乘法**

数学表达：
$$y(\tau, \nu) = h(\tau, \nu) \cdot x(\tau, \nu)$$

其中 $h(\tau, \nu)$ 是信道，$x(\tau, \nu)$ 是发送符号，$y(\tau, \nu)$ 是接收符号。

### 6.3 所有QAM符号经历相同的信道（不变性）

这是OTFS最关键的性质：**时不变性**

- 在延迟-多普勒域中，**每一个QAM符号都经历完全相同的信道响应**
- 这意味着接收端可以进行**单抽头均衡**（single-tap equalization）
- 彻底消除了OFDM中因频率选择性衰落导致的符号间干扰

### 6.4 OTFS vs OFDM 对比

| 特性 | OFDM | OTFS |
|------|------|------|
| 符号放置位置 | 时频网格 | 延迟-多普勒网格 |
| 信道作用形式 | 时频卷积（复杂） | 复增益乘法（简单） |
| 符号间干扰 | 存在（频率选择性衰落） | 不存在 |
| 均衡复杂度 | 高（多抽头） | 低（单抽头） |
| 导频开销 | 高 | 低（利用信道稀疏性） |

### 6.5 小结

OTFS通过在延迟-多普勒域发送QAM符号，实现了：

1. **信道估计简化** - 稀疏信道只需少量导频
2. **均衡简化** - 单抽头均衡替代复杂的多抽头均衡
3. **性能提升** - 消除符号间干扰，提升误码性能
4. **移动性支持** - 更好地处理高速移动场景的多普勒扩展

## 7. 思考题

### 思考题 1
假设一个无线信道有两条路径：
- 路径A：延迟 τ=0.5μs，多普勒 ν=100Hz
- 路径B：延迟 τ=2μs，多普勒 ν=-200Hz

请在延迟-多普勒网格上标出这两条路径的位置，并说明它们分别可能对应什么场景。

### 思考题 2
为什么说延迟-多普勒域是描述无线信道最"自然"的方式？请从物理角度解释。

### 思考题 3
假设一个运动的高铁列车作为反射器，请分析：
- 这会产生多大的多普勒频移？
- 如果列车速度为 350 km/h，载波频率为 5 GHz，多普勒频移是多少？

（提示：多普勒频移公式 $f_d = \frac{v}{c} f_c$）

### 思考题 4
OTFS相比OFDM的核心优势是什么？请从信道稀疏性和均衡复杂度两个角度分析。

### 思考题 5
在延迟-多普勒域中，信道冲激响应 h(τ, ν) 的稀疏性意味着什么？这对信道估计有什么影响？

---

**参考答案**

**思考题3答案**：
$v = 350 \text{ km/h} = 97.2 \text{ m/s}$，$f_c = 5 \text{ GHz}$，光速 $c = 3 \times 10^8 \text{ m/s}$

$$f_d = \frac{97.2}{3 \times 10^8} \times 5 \times 10^9 \approx 1620 \text{ Hz}$$

这么大的多普勒频移对于传统OFDM系统来说是一个严重挑战，但OTFS能够更好地处理这种情况。

---

**总结**

本notebook介绍了延迟-多普勒域表示的核心概念：

1. **延迟τ** 对应信号往返时间，即目标的距离
2. **多普勒ν** 对应运动目标引起的频率偏移
3. **h(τ, ν)** 是延迟-多普勒域的信道冲激响应，表示反射器集群的特性
4. **稀疏性** 是实际无线信道的重要特性
5. **OTFS** 将QAM符号放置在延迟-多普勒网格上，实现简单均衡和时不变性

这些概念为深入理解OTFS调制技术奠定了基础。